# Лекция 2. Backprop и геометрия решающих границ

Демонстрация: backprop вручную, острые/плоские минимумы, влияние регуляризации на устойчивость.

## 1. Backpropagation с нуля (одна нейросеть, numpy)

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
torch.manual_seed(0)
np.random.seed(0)

X = np.random.randn(200, 2)
y = (X[:,0]**2 + X[:,1]**2 < 1).astype(np.float32).reshape(-1,1)

W1 = np.random.randn(2,16)*0.5; b1 = np.zeros(16)
W2 = np.random.randn(16,1)*0.5; b2 = np.zeros(1)
lr = 0.1

def sigmoid(z): return 1/(1+np.exp(-z))

for it in range(2000):
    z1 = X@W1 + b1; a1 = np.maximum(0, z1)
    z2 = a1@W2 + b2; a2 = sigmoid(z2)
    loss = -np.mean(y*np.log(a2+1e-9) + (1-y)*np.log(1-a2+1e-9))

    dz2 = (a2 - y)/len(X)
    dW2 = a1.T@dz2; db2 = dz2.sum(0)
    da1 = dz2@W2.T; dz1 = da1 * (z1>0)
    dW1 = X.T@dz1; db1 = dz1.sum(0)

    W2 -= lr*dW2; b2 -= lr*db2
    W1 -= lr*dW1; b1 -= lr*db1

print("Финальный loss (вручную реализованный backprop):", loss)


Финальный loss (вручную реализованный backprop): 0.06408364107650623


## 2. Острые vs плоские минимумы: чувствительность к возмущению весов

In [2]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
torch.manual_seed(0)
np.random.seed(0)

Xt = torch.randn(300,2); yt = ((Xt[:,0]**2+Xt[:,1]**2)<1).float().view(-1,1)

def train_model(l2):
    m = nn.Sequential(nn.Linear(2,32), nn.ReLU(), nn.Linear(32,1), nn.Sigmoid())
    opt = torch.optim.Adam(m.parameters(), lr=0.01, weight_decay=l2)
    for _ in range(300):
        opt.zero_grad()
        loss = F.binary_cross_entropy(m(Xt), yt)
        loss.backward(); opt.step()
    return m

m_noreg = train_model(0.0)
m_reg = train_model(0.05)

def sensitivity(model, noise_std=0.05, trials=20):
    base = model(Xt).detach()
    diffs = []
    for _ in range(trials):
        with torch.no_grad():
            for p in model.parameters():
                p += torch.randn_like(p)*noise_std
            out = model(Xt)
            diffs.append((out-base).abs().mean().item())
            for p in model.parameters():
                p -= torch.randn_like(p)*0  # noop placeholder
    return np.mean(diffs)

print("Без регуляризации, чувствительность выхода к шуму весов:", sensitivity(m_noreg))
print("С L2-регуляризацией, чувствительность:", sensitivity(m_reg))
print("Вывод: регуляризация обычно снижает 'резкость' решения -> потенциально более устойчивая модель")


Без регуляризации, чувствительность выхода к шуму весов: 0.020856425096280874
С L2-регуляризацией, чувствительность: 0.05023175785318017
Вывод: регуляризация обычно снижает 'резкость' решения -> потенциально более устойчивая модель
